In [21]:
from pathlib import Path
import jsonlines as jsl
import pandas as pd 
from typing import Dict

dataset_type_2_dspath = {
    "gsm": "/Users/seonils/dev/rims_minimal/dataset/gsm8K_test.jsonl",
    "math": "/Users/seonils/dev/rims_minimal/dataset/MATH/MATH-full.jsonl",
    "ocw": "/Users/seonils/dev/rims_minimal/dataset/ocw/ocw_course.jsonl",
}

def insert_datatype_meta(records:list, dataset_type:str):
    for row in records:
        for qobj in row.keys():
            row[qobj]["meta"]["dataset_type"] = dataset_type
    return records


def insert_gt_answer_to_raw(records_raw: list, dataset_records: list)->list:
    # first align two records according to its question    
    def question_from_indiv_row(row: Dict = None) -> str:
        return row["CoTQueryObject"]["query_message"][-1]["content"].replace("Question: ", "").strip()
    df_raw = pd.DataFrame(records_raw)
    df_ds = pd.DataFrame(dataset_records).rename(columns = lambda x: "question" if x == "problem" else x) 
    df_raw["question"] = df_raw.apply(question_from_indiv_row, axis=1)
    print((df_raw.question != df_ds.question).sum())
    print(df_raw[df_raw.question != df_ds.question].question)
    print(df_ds[df_raw.question != df_ds.question].question)
    assert (df_raw.question == df_ds.question).all() 
    
    answers = df_ds.answer.tolist()
    for row_raw, answer in zip(records_raw, answers):
        for qobj in row_raw.keys():
            row_raw[qobj]["meta"]["gt_answer"] = answer
    return records_raw
    

# jsl of interests
for par in Path().glob("*"):
    dataset_type = par.suffix[1:]
    if dataset_type not in dataset_type_2_dspath:
        continue
    dataset_jsl = dataset_type_2_dspath[dataset_type]
    dataset_records = list(jsl.open(dataset_jsl))
    jslfs = list(par.glob("Meta-Llama*/n1*.jsonl"))
    for f in jslfs:
        print(f.name)
        records = list(jsl.open(f))
        # insert_datatype_meta(records, dataset_type) # done
        records = insert_gt_answer_to_raw(records, dataset_records)
        with jsl.open(f"{str(f)}_", "w") as writer:
            writer.write_all(records)
        
    

n1_baseline_raw_query_result.jsonl
0
Series([], Name: question, dtype: object)
Series([], Name: question, dtype: object)
n1_baseline_raw_query_result.jsonl
0
Series([], Name: question, dtype: object)
Series([], Name: question, dtype: object)
n1_baseline_raw_query_result.jsonl
0
Series([], Name: question, dtype: object)
Series([], Name: question, dtype: object)
